# MedTech: Clinical Data Exploration and SF 36 Regression

This notebook develops the original masterchart analysis into a readable workflow:
data-quality assessment, clinical and Prakriti feature engineering, PCA, K-means,
and comparison of linear regression, random forest, and XGBoost.

The original saved run used **226 records and 207 input columns**. Its aggregate
results are preserved in the [README](README.md) and [analysis notes](docs/analysis.md).
This cleaned workflow changes identifier handling and evaluation, so new results
must be reported separately. It does not contain the private source dataset.

**Reading order:** setup → aggregate audit → feature engineering → unsupervised
exploration → regression comparison → interpretation.

**Sharing rule:** keep record-level data local. Outputs are intentionally cleared;
run `python scripts/check_notebook.py --clear` before committing an executed notebook.

## 1. Setup

Start Jupyter from the repository root after installing `requirements.txt`.
Install `requirements-xgboost.txt` to include XGBoost. The original notebook used a
Python 3.10.19 kernel; other original package versions were not recorded.

In [ ]:
import os
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from medtech import MedTechFeatures

try:
    from xgboost import XGBRegressor
except ImportError:
    XGBRegressor = None
    print("XGBoost is unavailable. Install requirements-xgboost.txt to include it.")

RANDOM_STATE = 42
TARGET = "SF 36"
plt.rcParams.update({"figure.figsize": (9, 5), "axes.spines.top": False,
                     "axes.spines.right": False})

## 2. Load local data without displaying records

Place an authorized local CSV at `data/medtech_masterchart.csv`, or set
`MEDTECH_DATA_PATH` to your file. No personal machine path is embedded in the code.
Known identity fields (`Sno`, `Name`, `Place of birth`, `date`, `month`, `year`, `CR`)
are removed immediately. This removes known fields, not every possible indirect
identifier. All remaining record-level information still belongs in a private workspace.

In [ ]:
data_path = Path(os.environ.get("MEDTECH_DATA_PATH", "data/medtech_masterchart.csv"))
if not data_path.is_file():
    raise FileNotFoundError(
        "Input CSV is not available. See data/README.md and set MEDTECH_DATA_PATH "
        "or place the file at data/medtech_masterchart.csv."
    )

raw = pd.read_csv(data_path)
input_shape = raw.shape
raw.columns = raw.columns.str.strip()
if raw.columns.duplicated().any():
    raise ValueError("Duplicate column names after trimming whitespace; review the source CSV.")
identifier_columns = [c for c in raw if c.lower() in MedTechFeatures.IDENTIFIERS]
df = raw.drop(columns=identifier_columns).copy()
del raw
if TARGET not in df:
    raise ValueError("Required outcome column 'SF 36' is missing.")
print(f"Input dimensions: {input_shape[0]} rows × {input_shape[1]} columns")
print(f"Known identifier columns removed: {len(identifier_columns)}")

## 3. Audit structure and missingness

The original analysis inspected dimensions, dtypes, and the 30 columns with the
most missing values. These cells keep that audit while displaying only column-level
summaries. The original export included many `Unnamed:` fields; retain their source
names until a codebook explains what they represent.

In [ ]:
display(df.dtypes.astype(str).value_counts().rename("column_count").to_frame())
missingness = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_fraction": df.isna().mean(),
}).sort_values("missing_count", ascending=False, kind="stable")
display(missingness.head(30))

## 4. Define the outcome and hold-out split

`SF 36` is the spreadsheet's numeric outcome. Its scoring formula is not included
in the repository; do not assume it is a validated 0–100 domain score. Missing
outcomes are excluded rather than imputed. Features and outcome remain separate.

Split **80% training / 20% test**, seed **42**, before fitting any supervised
missingness filter, site vocabulary, imputer, or scaler. For 226 complete outcome
records this gives 180 training and 46 test records. Repeated records from one
person would require a grouped split; the source notebook did not establish whether
such repeats exist.

In [ ]:
outcome = pd.to_numeric(df[TARGET], errors="coerce")
valid_outcome = outcome.notna() & np.isfinite(outcome)
print(f"Records excluded for unavailable/non-finite outcomes: {(~valid_outcome).sum()}")
X = df.loc[valid_outcome].drop(columns=TARGET).copy()
y = outcome.loc[valid_outcome].astype(float)
if len(y) < 10:
    raise ValueError("Too few labeled records for this exploratory workflow.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(f"Training records: {len(X_train)}; test records: {len(X_test)}")

## 5. Feature engineering

The reusable `MedTechFeatures` transformer makes the original choices explicit:

- Remove raw `Q1`–`Q36`, the duplicate `Prakriti.1`, known identifiers, and the outcome.
- Lowercase/trim `Site`; categories with fewer than 5 training records map to `other`.
  One-hot encode with a dropped reference category. Unseen test sites use `other`.
- Parse TNM into numeric T, N, M categories; collapse letter substages. `Tumorsize`
  is an inherited feature name for **T category**, not a physical size measurement.
- Preserve the original reverse stage mapping: I→6, II→5, III→4, IVA→3, IVB→2, IVC→1.
- Encode V/P/K membership as `Vata_present`, `Pitta_present`, `Kapha_present`;
  retain the existing `Vatta`, `Pitta`, and `Kapha` numeric fields without overwriting.
- Drop all-empty training features and up to 30 ranked features with missing values;
  then mean-impute and standardize inside each fitted pipeline.

The primary feature set excludes `Value_Q1`–`Value_Q36`. A separate reconstruction
comparison includes them because that matches a key part of the original design.
Consult the [data dictionary](docs/data-dictionary.md) for meanings and unresolved codes.

In [ ]:
def make_pipeline(model, include_questionnaire_values=False):
    return Pipeline([
        ("features", MedTechFeatures(
            include_questionnaire_values=include_questionnaire_values,
            missing_columns_to_drop=30, min_site_count=5
        )),
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler()),
        ("model", model),
    ])

feature_audit = MedTechFeatures().fit(X_train)
print(f"Primary training feature count: {len(feature_audit.get_feature_names_out())}")
print(f"Columns removed by the training missingness filter: {len(feature_audit.dropped_columns_)}")
display(pd.DataFrame({"dropped_feature": feature_audit.dropped_columns_}))

## 6. PCA: explore structure in the feature matrix

This descriptive branch fits on all available labeled records and is not used by
the regression pipelines. It includes scored questionnaire values to retain the
original exploratory emphasis, but excludes the outcome and known identifiers.
Full-cohort fitting here does not feed learned parameters into the supervised split.

The original PCA requested 100 components. Cap this by the available rows and
features, then inspect cumulative explained variance. The original notebook did
not use PCA components for K-means or regression; this notebook preserves that separation.

In [ ]:
exploration = Pipeline([
    ("features", MedTechFeatures(include_questionnaire_values=True)),
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
])
exploration_scaled = exploration.fit_transform(X)
n_components = min(100, *exploration_scaled.shape)
pca = PCA(n_components=n_components, svd_solver="full")
X_pca = pca.fit_transform(exploration_scaled)

fig, ax = plt.subplots()
ax.plot(np.arange(1, n_components + 1), np.cumsum(pca.explained_variance_ratio_))
ax.set(xlabel="Number of principal components", ylabel="Cumulative explained variance",
       title="PCA on the exploratory feature matrix", ylim=(0, 1.02))
fig.tight_layout()
plt.show()

## 7. K-means: elbow inspection and five-cluster exploration

Evaluate k=1…9 on the standardized original feature space, then retain the
original five-cluster setting as an exploratory choice. The elbow is a heuristic;
the original run did not establish an optimal k or validate meaningful clinical
subgroups. Cluster counts below summarize the current run without printing membership.

In [ ]:
k_values = range(1, min(9, len(X)) + 1)
inertias = []
for k in k_values:
    estimator = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    estimator.fit(exploration_scaled)
    inertias.append(estimator.inertia_)

fig, ax = plt.subplots()
ax.plot(list(k_values), inertias, marker="o")
ax.set(xlabel="Number of clusters (k)", ylabel="Inertia", title="K-means elbow inspection")
fig.tight_layout()
plt.show()

kmeans = KMeans(n_clusters=5, random_state=RANDOM_STATE, n_init=10)
clusters = kmeans.fit_predict(exploration_scaled)
display(pd.Series(clusters).value_counts().sort_index().rename("record_count").to_frame())

## 8. Compare regression models in two feature settings

The original model families are retained: ordinary least squares, a 200-tree
random forest, and XGBoost. A mean predictor provides a simple reference. Random
seeds are explicit. No hyperparameter search is performed.

Both settings use the same raw split and fit separate training-only preprocessing:

1. **Without scored questionnaire items:** primary exploratory prediction setting.
2. **With scored questionnaire items:** reconstruction setting. If these items
   contribute to `SF 36`, performance may reflect rebuilding the score rather than
   predicting an independent outcome. The source scoring formula is needed to confirm this.

New MAE/MSE values use the **original spreadsheet outcome units**, unlike the
original saved run's standardized outcome. Do not compare the two error scales directly.

In [ ]:
models = {
    "Mean baseline": DummyRegressor(strategy="mean"),
    "Linear regression": LinearRegression(),
    "Random forest": RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=2),
}
if XGBRegressor is not None:
    models["XGBoost"] = XGBRegressor(random_state=RANDOM_STATE, n_jobs=2)

results = []
fitted_models = {}
for setting, include_values in [("Without scored questionnaire items", False),
                                 ("With scored questionnaire items", True)]:
    for name, model in models.items():
        pipeline = make_pipeline(clone(model), include_values)
        pipeline.fit(X_train, y_train)
        predictions = pipeline.predict(X_test)
        results.append({
            "Feature setting": setting, "Model": name,
            "MAE": mean_absolute_error(y_test, predictions),
            "MSE": mean_squared_error(y_test, predictions),
            "R2": r2_score(y_test, predictions),
        })
        fitted_models[(setting, name)] = pipeline

metrics = pd.DataFrame(results)
display(metrics.round(6))

## 9. Inspect random forest feature importance

Show the top 20 impurity-based importances for the primary feature setting.
This is an explanation of a fitted model, not evidence of a causal relationship.
Correlated features and different category structures can distort these rankings.

In [ ]:
forest = fitted_models[("Without scored questionnaire items", "Random forest")]
importance = pd.Series(
    forest.named_steps["model"].feature_importances_,
    index=forest.named_steps["features"].get_feature_names_out(),
).nlargest(20).sort_values()

fig, ax = plt.subplots(figsize=(12, max(8, len(importance) * 0.6)))
ax.barh(range(len(importance)), importance.to_numpy())
ax.set_yticks(range(len(importance)), [textwrap.fill(str(name), 48) for name in importance.index], fontsize=9)
ax.set(xlabel="Impurity-based importance", ylabel="Feature",
       title="Random forest: top 20 features without scored questionnaire items")
fig.tight_layout()
plt.show()

## 10. Original results and interpretation

These values were transcribed from the original notebook's saved evaluation
output, not recomputed by this cleanup. The original outcome was standardized
before the split; the original predictors included `CR` and `Value_Q1`–`Value_Q36`.

| Original model | MAE (standardized target) | MSE (standardized target²) | R² |
|---|---:|---:|---:|
| Linear regression | 0.002446 | 0.000236 | 0.999806 |
| Random forest (200 trees) | 0.278531 | 0.156415 | 0.870832 |
| XGBoost | 0.273296 | 0.146704 | 0.878851 |

The source notebook fit the imputer/scaler on all records before the hold-out
split. Its questionnaire-derived predictors also raise a score-reconstruction
question. Treat these as historical exploratory results, not validated clinical
performance. The cleaned workflow addresses preprocessing leakage, removes the
known record identifier, and makes the questionnaire comparison explicit.

The dataset is not distributed, so the historical scores cannot currently be
independently reproduced from this repository. New cluster interpretations,
cross-validation, grouped splitting where needed, and external validation remain
future work. See [analysis notes](docs/analysis.md) for the complete limitations.

## 11. Save only reviewed aggregate outputs

Generated files stay in ignored `outputs/`. Do not export row-level predictions,
cluster membership, or fitted artifacts for public sharing. Aggregate outputs
still need review before publication.

In [ ]:
# Optional local export of the current aggregate model comparison.
EXPORT_AGGREGATE_RESULTS = False
if EXPORT_AGGREGATE_RESULTS:
    output_dir = Path("outputs")
    output_dir.mkdir(exist_ok=True)
    metrics.to_csv(output_dir / "model_comparison.csv", index=False)